In [1]:
!pip install sdv

In [2]:
import pandas as pd
my_df = pd.read_csv("adult-data-census.csv",index_col=0)

In [3]:
df_reset = my_df.reset_index()


In [4]:
import numpy as np
df_reset = df_reset.replace('^ ', '', regex=True)
df_reset = df_reset.replace('?', np.nan)


In [5]:
null_values = df_reset.isnull().sum(axis=1)
for i in range(3):
    null_values = list(filter(lambda x: x > i, null_values))
    


In [6]:
df_reset.columns = list(map(lambda x: x.replace(" ", ""), df_reset.columns))

In [7]:
df_reset.columns = list(map(lambda x: x.replace("-", "_"), df_reset.columns))

In [8]:
df_reset_copy = df_reset.dropna()

In [9]:
columns_values = df_reset_copy.columns

In [10]:
is_col_continuos = []
is_col_categorical = []
is_col_categorical_many_examples = []
is_col_discrete = []
is_col_ordinal = []
import math

for elements_cols in df_reset_copy.columns:
    give_exception = False
    if ".float" in str(type(df_reset_copy.at[0, elements_cols])):
        give_exception = True
        for idx in range(len(df_reset_copy[elements_cols])):
            if 'float' in str(type(df_reset_copy.loc[idx, elements_cols])) and math.isnan(df_reset_copy.loc[idx, elements_cols]):
                continue
            give_exception = df_reset_copy.loc[idx, elements_cols] == float(int(df_reset_copy.loc[idx, elements_cols]))
            if give_exception == False:
                break
        if give_exception and df_reset_copy[elements_cols].nunique() < 20:
            is_col_categorical.append(elements_cols)
        elif give_exception and df_reset_copy[elements_cols].nunique() < 50:
            is_col_categorical_many_examples.append(elements_cols)
        else:
            is_col_continuos.append(elements_cols)
    
    elif df_reset_copy[elements_cols].nunique() < 20:
        is_col_categorical.append(elements_cols)
    elif df_reset_copy[elements_cols].nunique() < 50:
        is_col_categorical_many_examples.append(elements_cols)
    elif ".int" in str(type(df_reset_copy.at[0, elements_cols])):
        is_col_discrete.append(elements_cols)
    else:
        is_col_ordinal.append(elements_cols)


In [11]:
df_reset_copy_ = df_reset.copy()

Here we try to reduce the number of unique values from categorical columns, we will replace rare values with 'Others' to reduce balance the dataset.

In [12]:
from collections import Counter
for col_cat in is_col_categorical_many_examples:
    frequency_el = Counter(df_reset_copy[col_cat])
    d = dict(frequency_el)
    d = dict(sorted(d.items(), key=lambda key_val: key_val[1], reverse=True))
    val_len = len(df_reset_copy[col_cat])
    val = 0
    d_new = {}
    for idx, el in enumerate(d):
        val+= d[el]
        d_new[el] = d[el]
        if val > val_len * 0.98:
            break
    
    for nationality in df_reset_copy_[col_cat].unique():
        if nationality not in d_new.keys():
            df_reset_copy_.loc[df_reset_copy_[col_cat]==nationality, col_cat] = 'Others'
            df_reset_copy_.loc[df_reset_copy_[col_cat]==nationality, col_cat] = 'Others'
    is_col_categorical_many_examples.remove(col_cat)
    is_col_categorical.append(col_cat)


In [13]:
cols_to_ignore = []
for cols in df_reset_copy_.columns:
    columns_filtered = list(filter(lambda x: cols in x and cols!=x, df_reset_copy_.columns)) 
    if columns_filtered == []:
        continue
    for extr_column in columns_filtered: 
        extr_column_2 = extr_column
        extr_column = extr_column.replace(cols, "")
        if "num" in extr_column or "numeric" in extr_column:
            #df_reset_copy_ = df_reset_copy_.drop(cols, axis=1)
            is_col_ordinal.append(extr_column_2)
            is_col_categorical.remove(extr_column_2)
            cols_to_ignore.append(cols)
            is_col_categorical.remove(cols)
            break


In [14]:
is_col_OH =is_col_categorical.copy()
for col_val in is_col_categorical:
    if ".int" in str(type(df_reset_copy_.at[0, col_val])) or ".float" in str(type(df_reset_copy_.at[0, col_val])):
        is_col_OH.remove(col_val)
    

In [15]:
import re
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(sparse_output=False)
one_hot_encoded = encoder.fit_transform(df_reset_copy[is_col_OH])
one_hot_df = pd.DataFrame(one_hot_encoded,  columns=encoder.get_feature_names_out(is_col_OH))
df_reset_copy = pd.concat([df_reset_copy.drop(is_col_OH, axis=1), one_hot_df], axis=1)




!NEED TO OPTIMIZE HERE, MAYBE USE MATRIX MULTIPLICATION INSTEAD

In [16]:
# THESE TERMS ARE PROTECTED AND CLASSIFIED AS SENSITIVE UNDER THE US EEOC
list_of_sensitive_words = ['race', 'color', 'religion', 'sex', 'gender', 'age', 'disability', 'country', 'region', 'country_of_origin', 'native_country', 'salary']

In [17]:
from common_functions_f__ import generate_embedings_index
import numpy as np
embeddings_index =  generate_embedings_index()

In [18]:
import faiss
list_of_sensitive_embeddings = np.array([embeddings_index[word] for word in list_of_sensitive_words], dtype=np.float32, order='C')
faiss.normalize_L2(list_of_sensitive_embeddings)

In [19]:
from scipy.stats import chi2_contingency
correlated_values_2 = {}
correlated_values_3 = {}

highest_corr_values = []
correlated_columns = set()
value_cosine_similarity = {}
for idx_col_cat in range(len(is_col_categorical)):

    for idx_col_cat_2 in range(idx_col_cat+1, len(is_col_categorical)):
        col_cat = is_col_categorical[idx_col_cat]
        
        col_cat_2 = is_col_categorical[idx_col_cat_2]
        
        contingency_table = pd.crosstab(df_reset_copy_[col_cat], df_reset_copy_[col_cat_2])
        chi2_statistic, p_value, dof, expected = chi2_contingency(contingency_table)
        n = contingency_table.sum().sum()
        phi2 = chi2_statistic / n
        r, k = contingency_table.shape
        if n == 1:
            v = 0
        else: 
            phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
            k_corr = k - (k - 1) * (k - 2) / (n - 1)
            r_corr = r - (r - 1) * (r - 2) / (n - 1)
            if k_corr == 1 and r_corr == 1:
                v = 0
            elif k_corr != 1 and r_corr==1:
                v = np.sqrt(phi2corr/(k_corr-1))
            elif r_corr != 1 and k_corr == 1:
                v = np.sqrt(phi2corr/(r_corr-1))
            else:
                v = np.sqrt(phi2corr / min(k_corr - 1, r_corr - 1))
        if col_cat not in value_cosine_similarity.keys():
            if col_cat in embeddings_index.keys():
                embedding_word = np.array([embeddings_index[col_cat]], dtype=np.float32, order='C')
                faiss.normalize_L2(embedding_word)
                values = np.einsum('jk,lk->jl', embedding_word, list_of_sensitive_embeddings)
            else:
                sp_word = ""
                for i in range(len(col_cat)):
                    word = list(filter(lambda x: x.startswith(col_cat[0:i]) and x.endswith(col_cat[i+1:]), embeddings_index.keys()))
                    if len(word) > 0:
                        sp_word = word[0]
                        break

                embedding_word = np.array([embeddings_index[sp_word]], dtype=np.float32, order='C')
                faiss.normalize_L2(embedding_word)
                values = np.einsum('jk,lk->jl', embedding_word, list_of_sensitive_embeddings)
            values = np.max(values)
            value_cosine_similarity[col_cat] = values
        else:
            values = value_cosine_similarity[col_cat]

        if col_cat_2 not in value_cosine_similarity.keys():
            if col_cat_2 in embeddings_index.keys():
                embedding_word_2 = np.array([embeddings_index[col_cat_2]], dtype=np.float32, order='C')
                faiss.normalize_L2(embedding_word_2)
                values_2 = np.einsum('jk,lk->jl', embedding_word_2, list_of_sensitive_embeddings)
            else:
                sp_word = ""
                for i in range(len(col_cat)):
                    word = list(filter(lambda x: x.startswith(col_cat[0:i]) and x.endswith(col_cat[i+1:]), embeddings_index.keys()))
                    if len(word) > 0:
                        sp_word = word[0]
                        break

                embedding_word = np.array([embeddings_index[sp_word]], dtype=np.float32, order='C')
                faiss.normalize_L2(embedding_word)
                values_2 = np.einsum('jk,lk->jl', embedding_word, list_of_sensitive_embeddings)
            values_2 = np.max(values_2)
            value_cosine_similarity[col_cat_2] = values_2
        else:
            values_2 = value_cosine_similarity[col_cat_2]

        
        if col_cat == "income" or col_cat=="occupation":
            print("v=", v, "values=", values, " values_2=", values_2, "col_cat=", col_cat, "col_cat_2=", col_cat_2)
        
        if col_cat_2 == "income" or col_cat_2=="occupation":
            print("v=", v, "values=", values, " values_2=", values_2, col_cat, "col_cat_2=", col_cat_2)
        if v < 0.5 and (values > 0.5 or values_2 > 0.5):
            continue

        
        if v > 0.35:
            
            if (col_cat, col_cat_2) not in correlated_columns or (col_cat_2, col_cat) not in correlated_columns:
                correlated_columns.add((col_cat, col_cat_2))
            if col_cat in correlated_values_2.keys():
                list_cat = correlated_values_2[col_cat]
                list_cat_2 = correlated_values_3[col_cat]
                list_cat.append((float(v), col_cat_2))
                list_cat_2.append(col_cat_2)
                correlated_values_2[col_cat] = list_cat
                correlated_values_3[col_cat] = list_cat_2

            else:
                correlated_values_2[col_cat] = [(float(v), col_cat_2)]
                correlated_values_3[col_cat] = [col_cat_2]
            
            if col_cat_2 in correlated_values_2.keys():
                list_cat = correlated_values_2[col_cat_2]
                list_cat_2 = correlated_values_3[col_cat_2]
                list_cat.append((float(v), col_cat))
                list_cat_2.append(col_cat)
                correlated_values_2[col_cat_2] = list_cat
                correlated_values_3[col_cat_2] = list_cat_2
            else:
                correlated_values_2[col_cat_2] = [(float(v), col_cat)]
                correlated_values_3[col_cat_2] = [col_cat]
      

v= 0.21504335761931118 values= 0.123618186  values_2= 0.2911474 workclass col_cat_2= occupation
v= 0.16343725829197667 values= 0.123618186  values_2= 0.5092484 workclass col_cat_2= income
v= 0.12989358295622438 values= 0.32279527  values_2= 0.2911474 marital_status col_cat_2= occupation
v= 0.4471978508412036 values= 0.32279527  values_2= 0.5092484 marital_status col_cat_2= income
v= 0.1767990383051488 values= 0.2911474  values_2= 0.32621256 col_cat= occupation col_cat_2= relationship
v= 0.08020585167655005 values= 0.2911474  values_2= 0.99999994 col_cat= occupation col_cat_2= race
v= 0.43377337825761614 values= 0.2911474  values_2= 1.0000001 col_cat= occupation col_cat_2= sex
v= 0.34855268617063223 values= 0.2911474  values_2= 0.5092484 col_cat= occupation col_cat_2= income
v= 0.34855268617063223 values= 0.2911474  values_2= 0.5092484 occupation col_cat_2= income
v= 0.06408956867219309 values= 0.2911474  values_2= 1.0 col_cat= occupation col_cat_2= native_country
v= 0.4534156189553626 

In [20]:
from scipy.stats import chi2_contingency
from sklearn.preprocessing import LabelEncoder

cramer_values = {}
from scipy.stats import pearsonr
value_with_target = {}
is_col_numeric = is_col_discrete.copy()
is_col_numeric.extend(is_col_continuos)
for col_num_idx in range(len(is_col_numeric)-1):
    for col_num2_idx in range(col_num_idx+1, len(is_col_numeric)):
        col_num = is_col_numeric[col_num_idx]
        col_num2 = is_col_numeric[col_num2_idx]
        corr, _ = pearsonr(df_reset_copy_[col_num], df_reset_copy_[col_num2])
        if abs(corr) > 0.5:
            if col_num in correlated_values_3.keys():
                list_cat_2 = correlated_values_3[col_num]
                list_cat_2.append(col_num2)
                correlated_values_3[col_num] = list_cat_2

            else:
                correlated_values_3[col_num] = [col_num2]
            
            if col_num2 in correlated_values_3.keys():
                list_cat_2 = correlated_values_3[col_num2]
                list_cat_2.append(col_num)
                correlated_values_3[col_num2] = list_cat_2
            else:
                correlated_values_3[col_num2] = [col_num]
            



THERE WERE PROBLEMATIC ASSOCIATION BETWEEN SEX AND OCCUPATION, LATER FOR EXAMPLE, WE FOUND That in this dataset, there are no woman in the armed forces, but it is not an impossible situation like the "Unmarried" and "Husband" association, so we try to exclude moderate correlations associations related to sex or gender, in order to avoid bias in the dataset.

In [21]:
for key_ in correlated_values_2.keys():
    list_cat = correlated_values_2[key_]
    list_cat.sort(key=lambda x: x[0], reverse=True)
    correlated_values_2[key_] = list_cat

In [22]:
import statsmodels.api as sm
from statsmodels.formula.api import ols
for col_cat in is_col_categorical:
    is_col_numeric = is_col_discrete

    for col_num in is_col_numeric:
        
        model = ols(f'{col_num} ~ {col_cat}',data = df_reset_copy_).fit()
                
        anova_result = sm.stats.anova_lm(model, typ=2)
        eta_2 = anova_result['sum_sq'][col_cat]/(anova_result['sum_sq'][col_cat] + anova_result['sum_sq']['Residual'])

        if col_cat not in value_cosine_similarity.keys():
            if col_cat in embeddings_index.keys():
                embedding_word = np.array([embeddings_index[col_cat]], dtype=np.float32, order='C')
                faiss.normalize_L2(embedding_word)
                values = np.einsum('jk,lk->jl', embedding_word, list_of_sensitive_embeddings)
            else:
                sp_word = ""
                for i in range(len(col_cat)):
                    word = list(filter(lambda x: x.startswith(col_cat[0:i]) and x.endswith(col_cat[i+1:]), embeddings_index.keys()))
                    if len(word) > 0:
                        sp_word = word[0]
                        break

                embedding_word = np.array([embeddings_index[sp_word]], dtype=np.float32, order='C')
                faiss.normalize_L2(embedding_word)
                values = np.einsum('jk,lk->jl', embedding_word, list_of_sensitive_embeddings)
            values = np.max(values)
            value_cosine_similarity[col_cat] = values
        else:
            values = value_cosine_similarity[col_cat]

        if col_cat_2 not in value_cosine_similarity.keys():
            if col_cat_2 in embeddings_index.keys():
                embedding_word_2 = np.array([embeddings_index[col_cat_2]], dtype=np.float32, order='C')
                faiss.normalize_L2(embedding_word_2)
                values_2 = np.einsum('jk,lk->jl', embedding_word_2, list_of_sensitive_embeddings)
            else:
                sp_word = ""
                for i in range(len(col_cat)):
                    word = list(filter(lambda x: x.startswith(col_cat[0:i]) and x.endswith(col_cat[i+1:]), embeddings_index.keys()))
                    if len(word) > 0:
                        sp_word = word[0]
                        break

                embedding_word = np.array([embeddings_index[sp_word]], dtype=np.float32, order='C')
                faiss.normalize_L2(embedding_word)
                values_2 = np.einsum('jk,lk->jl', embedding_word, list_of_sensitive_embeddings)
            values_2 = np.max(values_2)
            value_cosine_similarity[col_cat_2] = values_2
        else:
            values_2 = value_cosine_similarity[col_cat_2]


    
        if (values > 0.5 or values_2 > 0.5) and eta_2 < 0.35:
            continue

        if eta_2 > 0.16:

            if col_cat in correlated_values_3.keys():
                list_cat_2 = correlated_values_3[col_cat]
                list_cat_2.append(col_num)
                correlated_values_3[col_cat] = list_cat_2

            else:
                correlated_values_3[col_cat] = [col_num]
            
            if col_num in correlated_values_3.keys():
                list_cat_2 = correlated_values_3[col_num]
                list_cat_2.append(col_cat)
                correlated_values_3[col_num] = list_cat_2
            else:
                correlated_values_3[col_num] = [col_cat]


The high correlations are related to columns which do not have missing values

There are low correlations between the columns in the dataset, which suggests that the each feature is not correlated with the other, so the inputting must be done in a generalised manner, or through intuition. So there is no need for eliminating columns.

In [23]:
are_missing_values = []
for idx, values in enumerate(df_reset.isnull().sum()):
    if values > 0:
        are_missing_values.append(df_reset.columns[idx])

In [24]:
null_values = df_reset_copy_.isnull().sum(axis=1)
null_values = list(map(lambda x: x[0], list(filter(lambda x: x[1] > i, zip(list(null_values.index), null_values)))))

df_reset_copy_ = df_reset_copy_.drop(null_values)

In [25]:
import warnings
warnings.filterwarnings(
    action='ignore', category=UserWarning, message=r"Boolean Series.*"
)


In [26]:
correlated_dict_ = set()
current_lists = []
correlated_elements_dict = {}
missing_pair_valid = set()
for val_key in correlated_values_3.keys():
    new_list = []
    new_list.append(val_key)
    for values in correlated_values_3[val_key]:
        ok = 0
        if (val_key, values) in correlated_dict_ or (values, val_key) in correlated_dict_:
            continue
        a = len(df_reset_copy_[val_key].unique())
        b = len(df_reset_copy_[values].unique())
        use_a = False
        use_b = False
        if a > b:
            use_a = a > b
        
        for dx in df_reset_copy_[val_key].unique():
            ok = 0
            for dy in df_reset_copy_[values].unique():
                
                if len(df_reset_copy_.loc[df_reset_copy_[val_key]==dx][df_reset_copy_[values]==dy]) < 0.01 * len(df_reset_copy_.loc[df_reset_copy_[val_key]==dx if a > b else df_reset_copy_[values]==dy]):
                    ok = 1
                    idx = df_reset_copy_[df_reset_copy_[val_key]==dx][df_reset_copy_[values]==dy].index
                    if len(list(idx)) > 0 and ((val_key, values) not in missing_pair_valid and (values, val_key) not in missing_pair_valid):
                        missing_pair_valid.add((val_key, values))

                    if a > b:
                        if len(list(idx)) > 0:
                            df_reset_copy_.loc[list(idx), val_key] = [np.nan] * len(list(idx))
                        if val_key not in are_missing_values:
                            are_missing_values.append(val_key)
                        
                    else:
                        if len(list(idx)) > 0:
                            df_reset_copy_.loc[list(idx), values] = [np.nan] * len(list(idx))
                        if values not in are_missing_values:
                            are_missing_values.append(values)
    
                if ok == 1:
                    break
            if ok == 1:
                break
        if ok == 1:
            new_list.append(values)
            correlated_dict_.add((val_key, values))
    current_lists.append(new_list)
        


In [27]:
for val_key, values in list(missing_pair_valid):
    for dx in df_reset_copy_[val_key].unique():
        for dy in df_reset_copy_[values].unique():
            if len(df_reset_copy_.loc[df_reset_copy_[val_key]==dx][df_reset_copy_[values]==dy]) < 0.01 * len(df_reset_copy_.loc[df_reset_copy_[val_key]==dx if a > b else df_reset_copy_[values]==dy]):
                idx = df_reset_copy_[df_reset_copy_[val_key]==dx][df_reset_copy_[values]==dy].index
                if a > b:
                    if len(list(idx)) > 0:
                        df_reset_copy_.loc[list(idx), val_key] = [np.nan] * len(list(idx))
                    
                else:
                    if len(list(idx)) > 0:
                        df_reset_copy_.loc[list(idx), values] = [np.nan] * len(list(idx))

 


In [28]:
df_reset_copy_['relationship'][7109]

nan

In [29]:
import math
from numpy.random import choice

from collections import Counter
associate_combined_values_with_input_values = {}
set_combined_values_with_input_values = set()
for col_ in are_missing_values:
    list_val = []
    for idx in df_reset_copy_.index[df_reset_copy_[col_].isnull()]:
        if "str" in str(type(df_reset_copy_[col_][idx])):
            continue
        if math.isnan(df_reset_copy_[col_][idx]):
            combination_ = ""
            relevant_columns = ""
            if col_ in correlated_values_2.keys():
                for col_2 in correlated_values_2[col_]:
                    if ("str" in str(type(df_reset_copy_[col_2[1]][idx])) or math.isnan(df_reset_copy_[col_2[1]][idx])==False):
                        if combination_ != "":
                            combination_+=";"+str(df_reset_copy_[col_2[1]][idx])
                            relevant_columns+=";"+col_2[1]
                        else:
                            combination_=col_+";"+str(df_reset_copy_[col_2[1]][idx])
                            relevant_columns=col_2[1]

                if combination_ not in associate_combined_values_with_input_values.keys():
                    df_new = df_reset_copy_.copy()
                    for cols_split in relevant_columns.split(";"):
                        df_new_2 = df_new[df_new[cols_split]==df_new[cols_split][idx]].copy()
                        if len(df_new_2[col_].copy().dropna())!=0:
                            df_new = df_new_2
            else:
                combination_ = col_+";"
                df_new = df_reset_copy_.copy()

            if combination_ not in associate_combined_values_with_input_values.keys():
                data_last = Counter(df_new[col_].copy().dropna())

                our_value = data_last.most_common(1)[0][0]
                associate_combined_values_with_input_values[combination_] = our_value
            else:
                our_value = associate_combined_values_with_input_values[combination_]
            list_val.append(our_value)

    df_reset_copy_.loc[list(df_reset_copy_.index[df_reset_copy_[col_].isnull()]), col_] = list_val
                
        


In [30]:
for curr_list_idx in range(len(current_lists)-1):
    for curr_list_idx_2 in range(curr_list_idx+1, len(current_lists)):
        common_lists_1 = current_lists[curr_list_idx]
        common_lists_2 = current_lists[curr_list_idx_2]

        common_lists_1.extend(common_lists_2)
        if len(set(common_lists_1)) != len(common_lists_1):
            current_lists[curr_list_idx] = list(set(common_lists_1))
            current_lists[curr_list_idx_2] = []


In [31]:
current_lists = list(filter(lambda x: len(x) > 1, current_lists))

In [32]:
for cols in cols_to_ignore:
    df_reset_copy_ = df_reset_copy_.drop(cols, axis=1)

In [33]:
df_reset_copy_.to_csv("adult-data-census-complete.csv", index=False)

In [34]:
from sdv.io.local import CSVHandler

connector = CSVHandler()

In [35]:
data = connector.read(
    folder_name='',
    file_names=['adult-data-census-complete.csv'],
    read_csv_parameters={
        'parse_dates': False,
        'encoding':'latin-1'
    }
)

In [37]:
from sdv.metadata import Metadata

metadata = Metadata.detect_from_dataframes(data)

metadata.visualize()
metadata.update_column(
    column_name='education_num',
    sdtype='numerical',
    table_name='adult-data-census-complete'
)

metadata.validate()
metadata.save_to_json('metadata_.json')

In [39]:
from sdv.single_table import GaussianCopulaSynthesizer
from sdv.single_table import TVAESynthesizer
from sdv.cag import FixedCombinations

synthesizer = GaussianCopulaSynthesizer(metadata)
constrain_list = []
for list_of_constraint in current_lists:
    constraint_ = FixedCombinations(
        column_names=list_of_constraint
    )
    constrain_list.append(constraint_)
synthesizer.add_constraints(constrain_list)
synthesizer.fit(data['adult-data-census-complete'])
synthetic_data = synthesizer.sample(num_rows=100)

In [40]:
synthetic_data[synthetic_data['relationship']=='Wife'][synthetic_data['sex']=='Male']

,age,workclass,fnlwgt,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
